# BÀI THỰC HÀNH 2 - BÀI 5a (TRANG 45)
## KHAI THÁC LUẬT KẾT HỢP DỮ LIỆU NƯỚC ĐỨC (GERMANY)
- **Học viên:** Nguyễn Minh Vương
- **MSSV:** 6451071090
- **Môn học:** Khai thác dữ liệu
- **Yêu cầu:** Thực hiện khai thác tập phổ biến và luật kết hợp cho nước Đức 'Germany' với `min_sup = 5%` và `min_conf = 50%`

In [ ]:
# Cài đặt thư viện nếu chạy trên Google Colab
# !pip install pandas numpy matplotlib seaborn mlxtend openpyxl gdown

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
%matplotlib inline

# Tự động tải thư mục dataset từ link Google Drive
if not os.path.exists("dataset") or len(os.listdir("dataset")) == 0:
    try:
        import gdown
        drive_folder_url = "https://drive.google.com/drive/folders/1rv5SiO5zFlU-cyEXul09OpCvTXrdjG1m"
        print("Đang tải dataset từ Google Drive...")
        gdown.download_folder(drive_folder_url, output="dataset", quiet=False, use_cookies=False)
        print("✅ Tải dataset thành công!")
    except Exception as e:
        print("Lưu ý: Nếu chưa có thư viện gdown, hãy chạy lệnh: !pip install gdown")

print("Nguyễn Minh Vương - 6451071090")


### 1. Đọc dữ liệu và tiền xử lý làm sạch dữ liệu ban đầu

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

df = pd.read_excel('dataset/Online Retail.xlsx')

# Tiền xử lý dữ liệu
df['Description'] = df['Description'].str.strip()
df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
df['InvoiceNo'] = df['InvoiceNo'].astype('str')
df = df[~df['InvoiceNo'].str.contains('C')]

print(f"Kích thước tập dữ liệu sau khi làm sạch: {df.shape[0]} dòng, {df.shape[1]} cột")
display(df.head())


### 2. Lọc dữ liệu khách hàng nước Đức ('Germany') và tạo ma trận One-hot encoding giỏ hàng

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

# Gom nhóm theo hóa đơn và mặt hàng của nước Đức 'Germany'
basket_germany = (df[df['Country'] == "Germany"]
                  .groupby(['InvoiceNo', 'Description'])['Quantity'].sum()
                  .unstack().reset_index().fillna(0)
                  .set_index('InvoiceNo'))

# Hàm chuẩn hóa sang One-hot encoding (0 hoặc 1)
def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1

basket_germany = basket_germany.map(encode_units) if hasattr(basket_germany, 'map') else basket_germany.applymap(encode_units)

# Xóa cột cước phí POSTAGE nếu có
if 'POSTAGE' in basket_germany.columns:
    basket_germany.drop('POSTAGE', inplace=True, axis=1)

print(f"Ma trận giỏ hàng thị trường Đức: {basket_germany.shape[0]} hóa đơn, {basket_germany.shape[1]} mặt hàng")
display(basket_germany.iloc[:5, :8])


### 3. Tìm tập phổ biến bằng giải thuật Apriori và FP-Growth với min_sup = 5%

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

# Áp dụng Apriori với min_sup = 5% (0.05)
itemsets_germany_apriori = apriori(basket_germany, min_support=0.05, use_colnames=True)
print(f"Số lượng tập phổ biến tìm được bằng Apriori (min_sup=5%): {len(itemsets_germany_apriori)}")

# Áp dụng FP-Growth để đối chiếu
itemsets_germany_fpgrowth = fpgrowth(basket_germany, min_support=0.05, use_colnames=True)
print(f"Số lượng tập phổ biến tìm được bằng FP-Growth (min_sup=5%): {len(itemsets_germany_fpgrowth)}\n")

print("10 tập mục phổ biến đầu tiên:")
display(itemsets_germany_apriori.head(10))


### 4. Tạo luật kết hợp với min_conf = 50% (0.5) và in ra các luật

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

# Sinh luật kết hợp với độ tin cậy tối thiểu 50%
rules_germany = association_rules(itemsets_germany_apriori, metric="confidence", min_threshold=0.5)

print(f"Tổng số luật kết hợp tìm được cho thị trường Đức: {len(rules_germany)}\n")

rules_disp = rules_germany.copy()
rules_disp['antecedents'] = rules_disp['antecedents'].apply(lambda x: ', '.join(list(x)))
rules_disp['consequents'] = rules_disp['consequents'].apply(lambda x: ', '.join(list(x)))

for i in range(len(rules_disp)):
    print(f"Luật {i+1}: {rules_disp.loc[i, 'antecedents']}  ==>  {rules_disp.loc[i, 'consequents']}")
    print(f"         Support: {rules_disp.loc[i, 'support']:.4f} | Confidence: {rules_disp.loc[i, 'confidence']:.4f} | Lift: {rules_disp.loc[i, 'lift']:.4f}\n")


### 5. Vẽ biểu đồ phân tán (scatter plot) biểu diễn các luật kết hợp và đưa ra nhận xét

In [ ]:
print("Nguyễn Minh Vương - 6451071090")

plt.figure(figsize=(8, 6))
plt.title('Đồ thị phân tán các luật kết hợp tại thị trường Đức (Germany)', fontsize=13)
plt.xlabel('Độ hỗ trợ (Support)', fontsize=11)
plt.ylabel('Độ tin cậy (Confidence)', fontsize=11)

sns.scatterplot(x='support', y='confidence', size='lift', hue='lift', data=rules_germany, palette='viridis', sizes=(50, 200))
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


> **Nhận xét Bài 5a:**
> - Tại thị trường Đức (Germany), với ngưỡng `min_sup = 5%` và `min_conf = 50%`, các luật kết hợp chủ yếu tập trung vào các mặt hàng bát đĩa, hộp đựng thực phẩm và đồ gia dụng đồng bộ (như bộ đồ tròn, đĩa sao/chấm bi).
> - Hệ số `Lift` của các luật này đều lớn hơn 1 rất nhiều (thường từ 2 đến 7), chứng tỏ sự xuất hiện của vế trái làm tăng đáng kể xác suất mua của vế phải, thể hiện thói quen mua sắm theo bộ của khách hàng Đức.